# ASI08 Cascading Failures — Upload Artifacts & Run Evaluation

**OWASP Category**: ASI08 — Cascading Failures | **Risk Severity**: High/Critical

**Mapped LLM Categories**: LLM05, LLM06, LLM10

**ASI08 tests for**:
- Pipeline cascade failure (upstream errors propagating downstream, poisoning outputs)
- Resource exhaustion amplification (infinite loops, exponential agent proliferation, queue saturation)
- Shared dependency failure (single point of failure affecting all pipeline stages simultaneously)

In [ ]:
%pip install okareo python-dotenv --quiet

In [1]:
import sys
from pathlib import Path
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))
from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver
from owasp.common import init_okareo, parse_artifact, build_target, SINGLE_TURN_DRIVER_TEMPLATE
okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")

✓ Okareo SDK initialized (key: ...x9vEA)


In [2]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}
for p in sorted(scenarios_dir.glob("*.jsonl")):
    name = f"ASI08-{p.stem}"
    sc = okareo.upload_scenario_set(scenario_name=name, file_path=str(p))
    registered_scenarios[name] = sc
    print(f"  ✓ {name}")

registered_checks = {}
for p in sorted((CATEGORY_DIR / "checks").glob("*.md")):
    data = parse_artifact(p)
    r = okareo.create_or_update_check(
        name=data["name"], description=data["description"],
        check=ModelBasedCheck(prompt_template=data["prompt_template"], check_type=CheckOutputType.PASS_FAIL),
    )
    registered_checks[data["name"]] = r.id
    print(f"  ✓ {data['name']}")

  ✓ ASI08-pipeline-cascade-failure
  ✓ ASI08-resource-exhaustion-amplification
  ✓ ASI08-shared-dependency-failure
  ✓ ASI08-cascade-failure-detector


In [ ]:
target = build_target(CATEGORY_DIR)
CASCADE_CHECK = "ASI08-cascade-failure-detector"
st_driver = Driver(name="pass-through", prompt_template=SINGLE_TURN_DRIVER_TEMPLATE, temperature=0)

all_results = {}
for sname, sc in registered_scenarios.items():
    if sname == "ASI-08-pipeline-cascade-failure":
        try:
            run = okareo.run_simulation(
                target=target, driver=st_driver,
                name=f"ASI08 — {sname}",
                api_key=OKAREO_API_KEY, first_turn="target",
                scenario=sc, max_turns=1, checks=[CASCADE_CHECK],
            )
            all_results[sname] = run
            print(f"  ✓ {sname}: {getattr(run, 'app_link', run.id)}")
        except Exception as e:
            all_results[sname] = None; print(f"  ✗ {sname}: {e}")
    else:
        print(f"  ○ {sname}: Skipped")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nASI08 Complete — Total: {len(all_results)} | Errors: {errors}")

  ✓ ASI08-pipeline-cascade-failure: http://localhost:3000/project/acfafd81-c856-4aac-887a-d5ea89a87335/eval/2d9184ad-2792-4f26-8d97-7fa75ec44339
  ✓ ASI08-resource-exhaustion-amplification: http://localhost:3000/project/acfafd81-c856-4aac-887a-d5ea89a87335/eval/aba1de97-5a08-4286-96eb-12e8aa674d21
  ✓ ASI08-shared-dependency-failure: http://localhost:3000/project/acfafd81-c856-4aac-887a-d5ea89a87335/eval/1e998588-a7f1-4a0e-80ed-e2d1f0c1c828

ASI08 Complete — Total: 3 | Errors: 0
